In [4]:
import pandas as pd
df = pd.read_csv("milestone1_jestavi.csv")

In [5]:
df.info()
df.describe()
df.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125 entries, 0 to 124
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          125 non-null    int64  
 1   contract_id         125 non-null    int64  
 2   raw_text            125 non-null    object 
 3   apr                 125 non-null    float64
 4   term_months         125 non-null    int64  
 5   monthly_payment     125 non-null    int64  
 6   penalty_clause      93 non-null     object 
 7   recommended_action  125 non-null    object 
 8   risk_flag           125 non-null    object 
dtypes: float64(1), int64(4), object(4)
memory usage: 8.9+ KB


,Unnamed: 0,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
25,25,26,This contract between Prime Auto and Alex Kim ...,3.34,24,1080,NaN,Approve,High
60,60,61,This contract between Prime Auto and Sara Lope...,10.84,48,612,NaN,Reject,Low
120,120,121,This contract between CarHub Leasing and John ...,14.31,24,665,Late fee $25,Review manually,Medium
5,5,6,This contract between National Motors and Rahu...,6.99,48,818,NaN,Approve,High
56,56,57,This contract between CarHub Leasing and Emily...,14.53,48,239,Late fee $25,Reject,Medium


In [6]:
def parse_contract(text):
    result = {}

    # Extract APR
    import re
    apr_match = re.search(r'APR (\d+\.?\d*)%', text)
    result['apr_extracted'] = float(apr_match.group(1)) if apr_match else None

    # Extract term
    term_match = re.search(r'(\d+) months', text)
    result['term_extracted'] = int(term_match.group(1)) if term_match else None

    # Extract payment
    payment_match = re.search(r'monthly payment \$?(\d+)', text)
    result['monthly_payment_extracted'] = int(payment_match.group(1)) if payment_match else None

    # Extract penalty
    penalty_match = re.search(r'(Late fee \$\d+|Early termination fee \$\d+|None)', text)
    result['penalty_extracted'] = penalty_match.group(1) if penalty_match else None

    return result

parsed = df['raw_text'].apply(parse_contract)
parsed_df = pd.DataFrame(parsed.tolist())
parsed_df.head()

,apr_extracted,term_extracted,monthly_payment_extracted,penalty_extracted
0,10.49,24,959,None
1,3.78,24,804,Early termination fee $300
2,5.41,24,774,Late fee $50
3,5.26,48,1028,Late fee $25
4,5.97,36,1180,None


In [7]:
def risk_rule(row):
    if row['apr'] > 10:
        return "HIGH"
    if row['monthly_payment'] > 900:
        return "MEDIUM"
    return "LOW"

df['rule_based_risk'] = df.apply(risk_rule, axis=1)
df[['raw_text', 'rule_based_risk']].head()

,raw_text,rule_based_risk
0,This contract between Prime Auto and Emily Sto...,HIGH
1,This contract between ABC Motors and Priya Sha...,LOW
2,This contract between DriveEasy Finance and Sa...,LOW
3,This contract between National Motors and Alex...,MEDIUM
4,This contract between National Motors and John...,MEDIUM


In [9]:
final_output = df[['contract_id',
                   'raw_text',
                   'apr',
                   'term_months',
                   'monthly_payment',
                   'penalty_clause',
                   'recommended_action',
                   'risk_flag']]

In [10]:
final_output.to_csv("milestone1_jestavi.csv")